# AI Engineering Interview Prep
## Section: Vector Databases & Embeddings

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


---
## 🗄️ Section 6 — Vector Databases and Embeddings

### Q1. What are embeddings in the context of AI engineering?
**A:** Embeddings are dense numerical vectors (arrays of floats) that represent semantic meaning. Two pieces of text with similar meaning produce vectors that are geometrically close (high cosine similarity).

Why they matter for AI engineering:
- Enable semantic search (find "murder provisions" even if the query is "killing laws")
- Foundation of RAG systems — documents are stored as embeddings, queries are embedded and matched
- Used for classification, clustering, anomaly detection, recommendation

Types:
- **Word embeddings**: word2vec, GloVe — one vector per word
- **Sentence embeddings**: sentence-transformers — one vector per sentence/paragraph
- **Image embeddings**: ViT, CLIP — one vector per image
- **Multi-modal embeddings**: CLIP — text and image in the same vector space

🏷️ *FAISS stores embeddings for all Nyaya-Pro legal documents. Query "bail in murder case" embeds to a vector that's closest to relevant BNS sections — not just keyword matches.*


### Q2. How do embedding models convert text to vectors?
**A:**
1. **Tokenise** the input text into sub-word tokens
2. **Embed** each token via a lookup table → sequence of token vectors
3. **Transformer encoder** processes all tokens in parallel with bidirectional self-attention — each token sees all other tokens
4. **Pooling** — aggregate the token representations into a single vector:
   - Mean pooling: average of all token vectors (most common)
   - CLS token: use the [CLS] special token's representation (BERT-style)
5. **Normalise** to unit length (for cosine similarity)

The model is trained so that text with similar meaning produces similar vectors (using contrastive learning — push similar pairs close, push random pairs far apart).

Output: a vector of 384-1536 floats representing the semantic content of the entire input text.


### Q3. What is the difference between sparse and dense embeddings?
**A:**
**Sparse embeddings (BM25, TF-IDF):**
- Vocabulary-sized vectors (50K+ dimensions)
- Most values are 0 — only matched keywords have non-zero weights
- Exact keyword matching — "Section 302" matches exactly
- Fast to compute, no GPU needed
- Misses semantic similarity ("killing" ≠ "murder" in sparse space)

**Dense embeddings (neural):**
- Compact vectors (384-1536 dimensions)
- Every dimension is non-zero — dense information encoding
- Semantic matching — "killing" ≈ "murder" in dense space
- Requires neural model for computation
- Misses exact matches for rare terms or identifiers

**Best practice:** Hybrid search — use both. Dense for semantic understanding, sparse for exact term matching. Combine with RRF for best results.


### Q4. Explain cosine similarity, dot product, and Euclidean distance for vector search.
**A:**
**Cosine similarity:** Measures the angle between vectors. Range [-1, 1]. 1 = identical direction, 0 = orthogonal, -1 = opposite. Ignores magnitude — only cares about direction. **Best for text embeddings** (normalised vectors).
```
cos_sim(A,B) = A·B / (|A| × |B|)
```

**Dot product:** A·B = |A||B|cos(θ). For normalised unit vectors, equals cosine similarity. Faster to compute (no normalisation step). Many models train to maximise dot product directly.

**Euclidean distance (L2):** Geometric distance between two points. Range [0, ∞). Sensitive to vector magnitude — works well for non-normalised vectors. Less popular for semantic similarity.

**In practice:** Normalise embeddings to unit length → cosine similarity = dot product. FAISS defaults to dot product (fastest). Pinecone uses cosine similarity by default.


### Q5. What is a vector database, and how does it differ from a traditional database?
**A:**
**Traditional DB (PostgreSQL, MySQL):**
- Stores structured rows and columns
- Queries: exact match, range, aggregation (`WHERE id = 123`)
- Index: B-tree, hash (exact lookups)
- No semantic understanding

**Vector DB (Pinecone, Qdrant, Weaviate, FAISS):**
- Stores embedding vectors + metadata
- Queries: nearest neighbour ("find vectors closest to this query vector")
- Index: ANN (Approximate Nearest Neighbour) — HNSW, IVF
- Semantic understanding through vector geometry

**Why you need a vector DB (not just SQL):**
- SQL databases can store vectors but have no efficient ANN index
- Full exact nearest-neighbour in SQL: O(N×d) — too slow for millions of vectors
- Vector DBs use HNSW/IVF for O(log N) ANN search at scale

Some databases now have vector support (pgvector for PostgreSQL) — suitable for small to medium scale.


### Q6. How do you choose the right embedding model for your use case?
**A:**
**Decision framework:**
1. **Task type** → check MTEB leaderboard for your task: retrieval, clustering, semantic similarity, classification
2. **Dimension trade-off**: 384-dim (fast/small/free) vs 1536-dim (better quality/slower/paid)
3. **Context length**: do your documents exceed 512 tokens? Use models with longer max length
4. **Multilingual need**: paraphrase-multilingual-MiniLM-L12-v2 or multilingual-e5 for non-English
5. **Latency constraint**: local sentence-transformers for low latency; API models if quality wins
6. **Cost**: self-hosted (GPU cost) vs API (per-token billing)

**My ranking for RAG:**
1. `text-embedding-3-large` (OpenAI) — best quality, paid
2. `all-mpnet-base-v2` — best free quality
3. `all-MiniLM-L6-v2` — fastest free option, good enough for most cases

🏷️ *Nyaya-Pro: `all-MiniLM-L6-v2` for embeddings — fast, free, good enough for legal section matching at 512-token chunk size.*


### Q7. What is embedding dimensionality, and how does it affect performance and cost?
**A:** Embedding dimensionality = the length of the vector. Common sizes: 384, 768, 1024, 1536, 3072.

**Higher dimensionality:**
- More expressive — can encode more semantic information
- Higher quality (usually)
- More memory: 1M vectors × 1536 dims × 4 bytes = 6GB
- Slower search (more distance computations)
- Higher indexing cost

**Lower dimensionality:**
- Faster and cheaper
- Lower quality
- 1M vectors × 384 dims × 4 bytes = 1.5GB — 4× smaller

**Matryoshka Embeddings (OpenAI text-embedding-3):** The embedding is trained so that the first N dimensions are already informative. You can use 256 or 512 dims from a 1536-dim model and get 90%+ of the quality at 6× less memory. Great for cost-quality trade-off.


### Q8. How do you handle embedding drift when the embedding model is updated?
**A:** Embedding drift = when you upgrade your embedding model, all existing vectors are from a different model and are incompatible with new query vectors — search quality crashes.

**Strategies:**

1. **Full re-index** (cleanest) — delete all old embeddings, re-embed all documents with new model. Expensive for large corpora.
2. **Dual index** — run old and new models in parallel; query both; migrate traffic gradually to new index after validation
3. **Blue-green index** — build the entire new index offline; switch over atomically when ready
4. **Compatibility testing** — before upgrading, test on a sample: does new model improve retrieval quality enough to justify migration cost?
5. **Versioned indices** — tag each chunk with its embedding model version; maintain separate indices per model version

Prevention: Pin your embedding model version in code. Never auto-upgrade embedding models without a re-indexing plan.


### Q9. What are multi-modal embeddings, and how are they generated?
**A:** Multi-modal embeddings map different data types (text, image, audio) into the **same vector space** so that semantically related items across modalities are close together.

**Example (CLIP by OpenAI):**
- Text encoder: transformer → text vector
- Image encoder: ViT → image vector
- Trained with contrastive loss: pairs like (photo of a cat, "a cat") are pushed close; random pairs pushed far apart

**Result:** You can search for images using text queries and vice versa — they're in the same space.

**Applications:**
- Image search with text queries ("find photos of cats in sunlight")
- Video search with text
- Cross-modal RAG (retrieve relevant images when answering text questions)
- Zero-shot image classification

**Other multi-modal embedding models:** ImageBind (Meta) — can embed text, image, audio, video, IMU, depth all in one space.


### Q10. How do you index and query multi-tenant data in a vector database?
**A:**
**Multi-tenant** = multiple users/organisations, each with their own private data that should never be accessible to others.

**Approaches:**
1. **Namespace per tenant** (Pinecone): Each tenant gets its own isolated namespace; queries are scoped to the namespace automatically
2. **Metadata filtering** (most DBs): Tag every vector with `tenant_id`; filter at query time: `filter={"tenant_id": "acme_corp"}`
3. **Separate collections/indexes** (Qdrant, Chroma): Each tenant gets a dedicated collection. Strongest isolation; highest resource cost.
4. **Hybrid** (recommended for scale): Namespace per large customer, metadata filter for smaller customers who share a collection

**Security considerations:**
- Never rely on application-level filtering only — enforce at the DB query level
- Audit log all cross-tenant queries
- Encrypt stored vectors per tenant if data is sensitive (regulatory requirement)


### Q11. What is quantisation of embeddings, and how does it reduce storage costs?
**A:** Quantisation reduces the precision of embedding values, using fewer bits per number.

**Types:**
| Type | Bits/dim | Size reduction | Quality loss |
|------|---------|---------------|-------------|
| Float32 (default) | 32 | 1× | None |
| Float16 | 16 | 2× | Negligible |
| Int8 (SQ8) | 8 | 4× | ~1-2% |
| Binary | 1 | 32× | ~5-10% |

**Binary quantisation** is extreme but effective: each float → 1 bit (positive or negative). 1M vectors × 1536 dims: 6GB → 187MB. You lose some quality but a re-ranking step with full-precision vectors recovers most of it (Matryoshka approach).

**Practical recommendation:** Use FP16 by default (2× savings, zero quality loss). Use INT8 for very large indices. Binary only if storage is severely constrained and you have a quality-preserving reranker.


### Q12. How do you benchmark and evaluate embedding model quality?
**A:**
**Retrieval-specific evaluation:**
1. **Build a test set** — 100-200 (query, relevant_document) pairs for your specific domain
2. **Metrics**: NDCG@k, MRR@k, Hit Rate@k — measure how often the right document appears in top-k results
3. **Test multiple models** — run your test set against 3-5 candidate embedding models
4. **Measure latency** — embed 1000 queries and time it; cost-quality trade-off

**Standard benchmarks:**
- **MTEB** (Massive Text Embedding Benchmark) — 56 tasks across 8 categories. The go-to leaderboard.
- **BEIR** — diverse retrieval benchmark across 18 domains

**Domain-specific evaluation:**
General benchmarks may not reflect your domain's performance. Always build a domain-specific test set with real user queries from your application.

🏷️ *For Nyaya-Pro, I built a 100-pair test set of legal queries + relevant BNS sections. Compared MiniLM-L6 vs mpnet-base; MiniLM was 90% of mpnet quality at 4× the speed — chose MiniLM.*


### Q13. What is the role of metadata in vector databases?
**A:** Metadata = structured fields stored alongside each vector, used for filtering, sorting, and attribution.

**Common metadata fields:**
```json
{
  "source_document": "BNS_2023.pdf",
  "section_number": "302",
  "section_title": "Punishment for murder",
  "page_number": 47,
  "date_indexed": "2024-01-15",
  "document_type": "criminal_law",
  "jurisdiction": "India",
  "language": "en"
}
```

**Use cases:**
1. **Pre-filtering**: `filter={"document_type": "criminal_law"}` — search only criminal law sections
2. **Post-retrieval attribution**: Display source, section number, and page to users
3. **Access control**: `filter={"tenant_id": "user123"}` — enforce per-user data isolation
4. **Freshness filtering**: `filter={"date_indexed": {"$gte": "2024-01-01"}}` — only recent documents
5. **Faceted search**: Count results by category, type, date

🏷️ *Every Nyaya-Pro chunk has metadata: source_act (BNS/BNSS/Constitution), section_number, chapter. Filters ensure criminal queries only search criminal law chunks.*


### Q14. How do you handle large-scale vector search with billions of vectors?
**A:**
**Challenges at billions of vectors:** Memory (1B × 384 dims × 4 bytes = 1.5TB), search latency, index build time.

**Solutions:**
1. **Quantisation** — Binary quantisation reduces 1.5TB to 46GB. Use full-precision reranker on top-k candidates.
2. **IVF (Inverted File Index)** — cluster vectors into K centroids; search only the nearest clusters (e.g., top-100 of 1M clusters). O(K + n/K) instead of O(n).
3. **IVF-PQ (Product Quantisation)** — combine IVF with PQ compression. FAISS's IVF4096,PQ64 index handles billions with ~100ms search.
4. **Distributed sharding** — split vectors across multiple nodes; route queries to relevant shards; merge results
5. **GPU-accelerated search** — FAISS supports GPU for massive speedup on similarity computation
6. **Managed services** — Pinecone, Qdrant, Weaviate, Vespa handle the scaling complexity

**Practical recommendation:** For most AI applications (up to 100M vectors), Pinecone or Qdrant on a single node with HNSW is sufficient.


### Q15. What is hybrid search (combining keyword search with vector search)?
**A:** Hybrid search combines dense vector search (semantic) with sparse keyword search (BM25/TF-IDF) for better retrieval coverage.

**Why combine?**
- Vector search: "What are the murder provisions?" → correctly finds Section 302 BNS semantically
- Keyword search: "Section 302 BNS" → exact citation match (vector might miss exact phrasing)
- Hybrid: catches both semantic meaning AND exact matches

**Implementation:**
1. Run BM25 search → get ranked list of results
2. Run vector search → get ranked list of results
3. Combine with **Reciprocal Rank Fusion (RRF)**: `score = 1/(k + rank_sparse) + 1/(k + rank_dense)`
4. Re-rank combined results

**Tools:** Elasticsearch (BM25 + vector), Pinecone (sparse-dense hybrid), Weaviate (BM25 + vector built-in), Qdrant (sparse-dense support).

🏷️ *Nyaya-Pro v2 added BM25 via rank_bm25 library alongside FAISS; RRF combination improved exact citation recall by ~25%.*


### Q16. How do you fine-tune an embedding model for a specific domain?
**A:** General embedding models may not capture domain-specific similarity well ("bail" and "bond" aren't similar in general text but are in legal context).

**Fine-tuning approaches:**

1. **Contrastive fine-tuning (recommended):**
   - Build a dataset of (query, positive_passage, hard_negative_passage) triplets
   - Fine-tune with InfoNCE/triplet loss to push query close to positive, far from negatives
   - Even 1K high-quality triplets can significantly improve domain recall

2. **MNRL (Multiple Negatives Ranking Loss):**
   - Simpler: just (query, positive_passage) pairs
   - Other pairs in the batch serve as implicit negatives

3. **Domain adaptive pre-training:**
   - Further pre-train the BERT-style encoder on unlabelled domain text (no labels needed)
   - The model learns domain vocabulary and relationships

**Tools:** `sentence-transformers` training API, Hugging Face `datasets` + `Trainer`.

🏷️ *For Nyaya-Pro enhancement: would fine-tune MiniLM on legal QA pairs (BNS section, query that should match it) to improve domain-specific retrieval.*


### Q17. Your vector database for RAG is consuming too much memory. How do you reduce it?
**A:**
1. **Quantise vectors** — FP32 → FP16 (2× reduction, zero quality loss). INT8 for 4× reduction, small quality impact.
2. **Binary quantisation** — 32× reduction with reranker to recover quality
3. **Reduce embedding dimension** — switch from 1536-dim to 384-dim embedding model (4× reduction); check if retrieval quality is acceptable
4. **Matryoshka truncation** — use only first 512 dimensions of a Matryoshka-trained model (e.g., OpenAI text-embedding-3)
5. **PQ (Product Quantisation)** — FAISS IVF+PQ compresses each vector to a fraction of its original size
6. **Disk-based index** — DiskANN or FAISS's IVFPQ with on-disk storage — only centroid index in RAM, vectors on disk
7. **Prune the knowledge base** — remove redundant or outdated documents; fewer documents = less memory

🏷️ *Nyaya-Pro: FAISS with FP16 vectors (~50MB for legal corpus) fits entirely in-process on a 2GB RAM server — fast and cheap.*


### Q18. Your vector database cannot scale to millions of embeddings. How do you fix the bottleneck?
**A:**
1. **Switch from FAISS flat to HNSW** — flat index is O(N) exact search; HNSW is O(log N) ANN search
2. **IVF partitioning** — cluster into cells; search only nearby clusters. Set nprobe appropriately for recall-speed trade-off.
3. **Move to a managed vector DB** — Pinecone, Qdrant, Weaviate are built for scale; they handle sharding, replication, and index management automatically
4. **Horizontal sharding** — split the index across multiple machines; route queries based on ID hash or metadata cluster
5. **Pre-filter with metadata** — reduce search space before ANN search
6. **Dimension reduction** — PCA or embedding model with lower dimension → smaller index per vector
7. **Async batch ingestion** — index new vectors in background batches, not synchronously on each insert

For 10M+ vectors: Qdrant with HNSW on 32GB RAM handles this comfortably. For 100M+: distributed deployment or Pinecone/Weaviate enterprise.


### Q19. Your new embedding model has different dimensions from existing production vectors. How do you handle the mismatch?
**A:** You CANNOT compare vectors from different embedding models — they're in completely different vector spaces.

**Migration strategies:**

1. **Full re-index (cleanest):** 
   - Build new index with new model on all documents (offline)
   - Blue-green switch: when new index is ready, atomically switch all query traffic to new index
   - Delete old index

2. **Dual-model serving (zero-downtime):**
   - Run both models simultaneously
   - New queries embedded with new model → new index
   - Gradually migrate documents and cutover

3. **Proxy/adapter layer:**
   - Train a linear transformation to map old vectors → new space (works if models are similar architectures)
   - Approximate — quality loss

4. **Incremental migration:**
   - New documents → new model + new index
   - Old documents stay in old index temporarily
   - Query both, merge results (reranker handles the merge)
   - Gradually re-index old documents in batches

Always **pin your embedding model version** in code to prevent accidental upgrades.


### Q20. Your vector search returns irrelevant results despite high similarity scores. How do you fix it?
**A:** High similarity score + irrelevant results = the vectors don't encode what you think they encode.

**Root causes and fixes:**

1. **Embedding model isn't domain-specific** — general model doesn't understand domain terminology. Fix: fine-tune on domain data or use a domain-specific model.

2. **Chunks are too large** — embedding of a 2000-token chunk averages many topics; query matches a small part but the chunk appears relevant. Fix: smaller chunks (256-512 tokens).

3. **Query is too short/ambiguous** — short queries have weak embeddings. Fix: HyDE (embed a hypothetical answer instead of the question) or query expansion.

4. **Wrong similarity metric** — if vectors aren't normalised, dot product != cosine similarity. Fix: normalise all vectors and use cosine similarity.

5. **Low-quality embeddings** — some models produce embeddings that cluster around the origin, reducing discriminability. Fix: benchmark your embedding model.

6. **Missing reranker** — add cross-encoder reranking on top of vector search; cross-encoder is much more accurate.


### Q21. You deployed a new embedding model and search quality crashed overnight. How do you handle embedding drift?
**A:** Immediate actions:
1. **Rollback** — revert to the previous embedding model version immediately (the most important step)
2. **Revert traffic** — if query traffic was already switched, route back to old index
3. **Assess impact** — how many users affected? How severe is the quality drop?

**Root cause:** New embedding model produces vectors in a different space than the existing index — all similarity scores are meaningless.

**Fix forward:**
1. Build a new index using the new embedding model on ALL documents
2. Validate the new index against your retrieval test set
3. Only switch when new index quality is confirmed equal or better
4. Implement a canary deployment: route 5% of traffic to new index for 24 hours; compare quality metrics before full cutover

**Prevention:** Pin embedding model versions. Never auto-upgrade. Always test before deploying index changes.


### Q22. Your semantic search fails for short queries. How do you improve it?
**A:** Short queries (1-3 words) produce weak, ambiguous embeddings because the model has little context to work with.

**Fixes:**
1. **HyDE (Hypothetical Document Embedding)** — LLM generates a hypothetical ideal answer (paragraph) → embed the answer → search with the answer's embedding. Answer-to-document matching >> question-to-document.
2. **Query expansion** — LLM expands the short query with synonyms and related terms: "murder" → "murder, homicide, killing, unlawful death, Section 302" → embed expanded query
3. **Multi-query retrieval** — generate 3-5 paraphrases of the short query; retrieve for each; merge and deduplicate results
4. **Sparse + dense hybrid** — BM25 handles short exact-keyword queries well; combine with vector search
5. **Domain fine-tuning** — fine-tune the embedding model so "murder" and "Section 302" are close in vector space
6. **Step-back prompting** — generalise the query: "murder bail" → "provisions for bail in serious criminal offences"
